In [3]:
"""
tda_analysis_v3.py
==================
Refined Persistent Homology analysis of protein structures.

Two TDA methods retained:
  TORSION DATA:
    1. rips_eucl  – Vietoris-Rips on EUCLIDEAN distances in 9-D torsion space (ripser)
    2. alpha_pca  – Alpha Complex on PCA(2-D) projection of torsion vectors    (GUDHI)

  COORD DATA:
    3. rips_ripser – Vietoris-Rips on 3-D Cα XYZ, Euclidean                   (ripser)
    4. alpha_3d    – Alpha Complex on 3-D Cα XYZ                               (GUDHI)

The raw extraction DataFrame (residue number, residue name, torsion angles,
Cα XYZ, SSR) is printed and returned alongside the upsampled blocks.

Outputs per block:
  • Persistence diagrams  (birth-death scatter) for each method × each PDB
  • Betti-1 curves (β₁ vs filtration value) for each method × each PDB

Final output:
  • residue_vs_betti_torsion.png
  • residue_vs_betti_coords.png
"""

# ─────────────────────────────────────────────────────────────────────────────
# Imports
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ExpSineSquared, WhiteKernel
from sklearn.decomposition import PCA

import rpy2.robjects as robjects
from rpy2.robjects import r, pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

import ripser
import gudhi as gd

import matplotlib
matplotlib.use("Agg")          # change to "TkAgg" / "Qt5Agg" for interactive display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import os

# ─────────────────────────────────────────────────────────────────────────────
# 1. Setup  – R / bio3d
# ─────────────────────────────────────────────────────────────────────────────
bio3d = importr("bio3d")

_R_FUNC = r"""
get_torsion_data <- function(pdb_id, chain_id = NULL) {
  if (is.null(chain_id) || is.na(chain_id)) { chain_id <- NULL }
  pdb <- read.pdb(pdb_id)
  available_chains <- unique(pdb$atom$chain)
  if (is.null(chain_id)) { chain_id <- available_chains[1] }

  pdb_selected <- trim.pdb(pdb, chain = chain_id)
  pdb_protein  <- trim.pdb(pdb_selected, "protein")
  torsion      <- torsion.pdb(pdb_protein)

  torsion_df <- data.frame(torsion$tbl, alpha = torsion$alpha, omega = torsion$omega)
  torsion_df[is.na(torsion_df)] <- 0

  transform_sigmoid <- function(angles) 1 / (1 + exp(-(angles / 60)))

  cols_to_fix <- c("phi","psi","chi1","chi2","chi3","chi4","chi5","omega","alpha")
  for (col in cols_to_fix) {
    if (col %in% colnames(torsion_df))
      torsion_df[[col]] <- transform_sigmoid(torsion_df[[col]])
  }

  row_names <- rownames(torsion_df)
  torsion_df$resno_int <- as.integer(sub("\\..*", "", row_names))

  # ── Residue name + chain from CA atoms ──────────────────────────────────
  ca_atoms <- pdb_protein$atom[pdb_protein$atom$elety == "CA",
                                c("resno","resid","chain","x","y","z")]
  torsion_df$resname <- ca_atoms$resid [match(torsion_df$resno_int, ca_atoms$resno)]
  torsion_df$chain   <- ca_atoms$chain [match(torsion_df$resno_int, ca_atoms$resno)]
  torsion_df$resname[is.na(torsion_df$resname)] <- "UNK"
  torsion_df$chain  [is.na(torsion_df$chain)]   <- chain_id

  # ── Composite residue label:  chain/resno/resname  (e.g. A/42/SER) ──────
  torsion_df$residue <- paste(torsion_df$chain,
                               torsion_df$resno_int,
                               torsion_df$resname, sep = "/")

  torsion_df$ca_x <- ca_atoms$x[match(torsion_df$resno_int, ca_atoms$resno)]
  torsion_df$ca_y <- ca_atoms$y[match(torsion_df$resno_int, ca_atoms$resno)]
  torsion_df$ca_z <- ca_atoms$z[match(torsion_df$resno_int, ca_atoms$resno)]
  torsion_df$ca_x[is.na(torsion_df$ca_x)] <- 0
  torsion_df$ca_y[is.na(torsion_df$ca_y)] <- 0
  torsion_df$ca_z[is.na(torsion_df$ca_z)] <- 0

  torsion_df$SSR <- "TL"
  if (!is.null(pdb$helix$start)) {
    for (i in 1:length(pdb$helix$start)) {
      idx <- which(torsion_df$resno_int >= pdb$helix$start[i] &
                   torsion_df$resno_int <= pdb$helix$end[i])
      if (length(idx) > 0) torsion_df$SSR[idx] <- "AH"
    }
  }
  if (!is.null(pdb$sheet$start)) {
    for (i in 1:length(pdb$sheet$start)) {
      idx <- which(torsion_df$resno_int >= pdb$sheet$start[i] &
                   torsion_df$resno_int <= pdb$sheet$end[i])
      if (length(idx) > 0) torsion_df$SSR[idx] <- "BS"
    }
  }
  return(torsion_df)
}
"""
r(_R_FUNC)
_get_torsion_data_r = robjects.globalenv["get_torsion_data"]

# ─────────────────────────────────────────────────────────────────────────────
# 2. Column definitions
# ─────────────────────────────────────────────────────────────────────────────
TORSION_COLS = ["phi","psi","chi1","chi2","chi3","chi4","chi5","omega","alpha"]
COORD_COLS   = ["ca_x","ca_y","ca_z"]
META_COLS    = ["residue","resno_int","SSR"]

# Two methods per data type
TORSION_METHODS = ["rips_eucl", "alpha_pca"]
COORD_METHODS   = ["rips_ripser", "alpha_3d"]

METHOD_LABELS = {
    "rips_eucl"  : "Vietoris-Rips\n(Euclidean 9-D)",
    "alpha_pca"  : "Alpha Complex\n(PCA 2-D)",
    "rips_ripser": "Vietoris-Rips\n(ripser, Euclidean)",
    "alpha_3d"   : "Alpha Complex\n(3-D XYZ)",
}

# Persistence thresholds for "significant" bars
THRESHOLDS = {
    "rips_eucl"  : 0.05,
    "alpha_pca"  : 0.05,
    "rips_ripser": 0.50,
    "alpha_3d"   : 0.50,
}

SSR_COLORS = {"AH": "#EF7C7C", "BS": "#5DC8C0", "TL": "#A8D8A8"}
PDB1_COLOR = "#2979FF"
PDB2_COLOR = "#FF6D00"

# ─────────────────────────────────────────────────────────────────────────────
# 3. Upsampling
# ─────────────────────────────────────────────────────────────────────────────

def upsample_torsion_block(block_df, ssr_type, multiplier=10):
    if len(block_df) < 2:
        return block_df[TORSION_COLS].reset_index(drop=True)
    orig_x = np.arange(len(block_df)).reshape(-1, 1)
    new_x  = np.linspace(0, len(block_df) - 1, len(block_df) * multiplier).reshape(-1, 1)
    if   ssr_type == "AH": kernel = 1.0 * ExpSineSquared(periodicity=3.6, length_scale=1.0)
    elif ssr_type == "BS": kernel = 1.0 * Matern(length_scale=1.0, nu=2.5)
    else:                  kernel = 1.0 * RBF(length_scale=0.5)
    gp = GaussianProcessRegressor(kernel=kernel + WhiteKernel(noise_level=1e-5))
    gp.fit(orig_x, block_df[TORSION_COLS].values)
    return pd.DataFrame(gp.predict(new_x), columns=TORSION_COLS)


def upsample_coord_block(block_df, multiplier=10):
    if len(block_df) < 2:
        return block_df[COORD_COLS].reset_index(drop=True)
    orig_x = np.arange(len(block_df)).reshape(-1, 1)
    new_x  = np.linspace(0, len(block_df) - 1, len(block_df) * multiplier).reshape(-1, 1)
    gp = GaussianProcessRegressor(
        kernel=1.0 * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-5))
    gp.fit(orig_x, block_df[COORD_COLS].values)
    return pd.DataFrame(gp.predict(new_x), columns=COORD_COLS)


# ─────────────────────────────────────────────────────────────────────────────
# 4. extract_blocks
#    Returns torsion_blocks, coord_blocks, residue_ranges,
#    AND the raw full DataFrame (residue, resname, torsion angles, XYZ, SSR).
# ─────────────────────────────────────────────────────────────────────────────

def extract_blocks(pdb_id, chain_id=None, multiplier=10):
    """
    Fetch torsion + coordinate data for a PDB entry, print the raw tables,
    and return upsampled blocks plus the raw DataFrame.

    Returns
    -------
    torsion_blocks : dict  {block_key → upsampled torsion DataFrame}
    coord_blocks   : dict  {block_key → upsampled coord DataFrame}
    residue_ranges : dict  {block_key → (start_res, end_res, ssr_type)}
    raw_df         : pd.DataFrame  — the full, unsampled per-residue table
                     columns: residue, resname, phi, psi, chi1..chi5, omega,
                              alpha, ca_x, ca_y, ca_z, SSR
    """
    r_chain = chain_id if chain_id is not None else robjects.NULL
    with localconverter(robjects.default_converter + pandas2ri.converter):
        torsion_df = _get_torsion_data_r(pdb_id, r_chain)

    # ── Raw DataFrame: keep everything neatly ordered ─────────────────────
    RAW_COL_ORDER = (
        ["residue", "resno_int"] + TORSION_COLS + COORD_COLS + ["SSR"]
    )
    raw_df = torsion_df[[c for c in RAW_COL_ORDER if c in torsion_df.columns]].copy()

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 260)

    print(f"\n{'='*90}")
    print(f"  RAW EXTRACTION TABLE  —  {pdb_id}")
    print(f"  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR")
    print(f"{'='*90}")
    print(raw_df.to_string(index=False))
    print(f"{'='*90}\n")

    # ── Build SSR blocks for upsampling ───────────────────────────────────
    torsion_df["block_id"] = (torsion_df["SSR"] != torsion_df["SSR"].shift()).cumsum()

    torsion_blocks = {}
    coord_blocks   = {}
    residue_ranges = {}

    for _, blk in torsion_df.groupby("block_id"):
        ssr   = blk["SSR"].iloc[0]
        s_res = int(blk["resno_int"].min())
        e_res = int(blk["resno_int"].max())
        key   = f"upsampled_{ssr}_{s_res}_{e_res}"

        torsion_blocks[key] = upsample_torsion_block(blk, ssr, multiplier)
        coord_blocks[key]   = upsample_coord_block(blk, multiplier)
        residue_ranges[key] = (s_res, e_res, ssr)

    return torsion_blocks, coord_blocks, residue_ranges, raw_df


# ─────────────────────────────────────────────────────────────────────────────
# 5. TDA Methods  — two methods per data type
# ─────────────────────────────────────────────────────────────────────────────

def _safe_arr(dgms_raw):
    """Convert GUDHI persistence list to numpy array safely."""
    if dgms_raw is None or len(dgms_raw) == 0:
        return np.empty((0, 2))
    arr = np.array(dgms_raw)
    if arr.ndim == 1:
        return np.empty((0, 2))
    return arr


# ── Torsion ──────────────────────────────────────────────────────────────────

def pers_rips_eucl(data, cols=TORSION_COLS):
    """Vietoris-Rips on Euclidean distances in 9-D torsion space (ripser)."""
    pts = data[cols].values
    if len(pts) < 3:
        return np.empty((0, 2))
    res = ripser.ripser(pts, maxdim=1, metric="euclidean")
    return _safe_arr(res["dgms"][1])


def pers_alpha_pca(data, cols=TORSION_COLS, n_comp=2):
    """Alpha Complex on PCA(2-D) projection of torsion data (GUDHI).
    Filtration values converted from α² → radius via sqrt."""
    pts = data[cols].values
    n   = len(pts)
    if n < max(n_comp + 1, 4):
        return np.empty((0, 2))
    pca   = PCA(n_components=min(n_comp, pts.shape[1], n - 1))
    pts2d = pca.fit_transform(pts)
    alpha = gd.AlphaComplex(points=pts2d.tolist())
    st    = alpha.create_simplex_tree()
    st.compute_persistence()
    raw = st.persistence_intervals_in_dimension(1)
    arr = _safe_arr(raw)
    if len(arr) == 0:
        return arr
    finite_mask = ~np.isinf(arr)
    return np.where(finite_mask, np.sqrt(np.maximum(arr, 0)), arr)


# ── Coord ────────────────────────────────────────────────────────────────────

def pers_rips_ripser_coords(data):
    """Vietoris-Rips on 3-D Cα XYZ, Euclidean (ripser)."""
    pts = data[COORD_COLS].values
    if len(pts) < 3:
        return np.empty((0, 2))
    res = ripser.ripser(pts, maxdim=1, metric="euclidean")
    return _safe_arr(res["dgms"][1])


def pers_alpha_3d(data):
    """Alpha Complex on 3-D Cα XYZ (GUDHI). Filtration → sqrt(α²)."""
    pts = data[COORD_COLS].values
    if len(pts) < 4:
        return np.empty((0, 2))
    alpha = gd.AlphaComplex(points=pts.tolist())
    st    = alpha.create_simplex_tree()
    st.compute_persistence()
    raw = st.persistence_intervals_in_dimension(1)
    arr = _safe_arr(raw)
    if len(arr) == 0:
        return arr
    finite_mask = ~np.isinf(arr)
    return np.where(finite_mask, np.sqrt(np.maximum(arr, 0)), arr)


# ── Dispatchers ──────────────────────────────────────────────────────────────

def compute_all_dgms_torsion(data):
    return {
        "rips_eucl": pers_rips_eucl(data, TORSION_COLS),
        "alpha_pca": pers_alpha_pca(data, TORSION_COLS),
    }


def compute_all_dgms_coords(data):
    return {
        "rips_ripser": pers_rips_ripser_coords(data),
        "alpha_3d"   : pers_alpha_3d(data),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 6. Betti-1 from diagram
# ─────────────────────────────────────────────────────────────────────────────

def betti1_from_dgms(dgms, threshold=None):
    """Max β₁ at any filtration value, after applying persistence threshold."""
    if len(dgms) == 0:
        return 0
    bars = dgms.copy()
    if threshold is not None:
        pers = bars[:, 1] - bars[:, 0]
        bars = bars[pers > threshold]
    if len(bars) == 0:
        return 0
    finite = bars[~np.isinf(bars).any(axis=1)]
    if len(finite) == 0:
        return 0
    all_vals = finite.flatten()
    xs     = np.linspace(all_vals.min(), all_vals.max(), 500)
    deaths = np.where(np.isinf(bars[:, 1]), xs[-1], bars[:, 1])
    curve  = [int(np.sum((bars[:, 0] <= x) & (x <= deaths))) for x in xs]
    return int(np.max(curve))


# ─────────────────────────────────────────────────────────────────────────────
# 7. Plotting helpers
# ─────────────────────────────────────────────────────────────────────────────

def _plot_pers_diagram(ax, dgms, title, color, threshold=None):
    """Persistence diagram (birth vs death scatter)."""
    ax.set_facecolor("#F7F9FC")
    ax.set_title(title, fontsize=8, fontweight="bold", pad=4)

    if len(dgms) == 0:
        ax.text(0.5, 0.5, "No bars", ha="center", va="center",
                transform=ax.transAxes, color="#888", fontsize=8)
        return

    bars = dgms.copy()
    if threshold is not None:
        bars = bars[(bars[:, 1] - bars[:, 0]) > threshold]

    finite = bars[~np.isinf(bars).any(axis=1)]
    inf_b  = bars[np.isinf(bars[:, 1])]

    if len(finite) > 0:
        ax.scatter(finite[:, 0], finite[:, 1],
                   c=color, alpha=0.75, s=28, zorder=3,
                   edgecolors="white", linewidth=0.4)
        mn = min(finite[:, 0].min(), finite[:, 1].min())
        mx = max(finite[:, 0].max(), finite[:, 1].max())
        ax.plot([mn, mx], [mn, mx], "k--", alpha=0.25, lw=0.8)

    if len(inf_b) > 0:
        y_top = finite[:, 1].max() * 1.12 if len(finite) > 0 else 1.0
        ax.scatter(inf_b[:, 0], [y_top] * len(inf_b),
                   c=color, marker="^", s=40, zorder=3,
                   edgecolors="white", linewidth=0.4, label="∞")
        ax.legend(fontsize=6, loc="upper left", framealpha=0.5)

    ax.set_xlabel("Birth", fontsize=7)
    ax.set_ylabel("Death", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.2, lw=0.5)


def _plot_betti_curve(ax, dgms, title, color, threshold=None):
    """β₁ curve (# alive bars vs filtration)."""
    ax.set_facecolor("#F7F9FC")
    ax.set_title(title, fontsize=8, fontweight="bold", pad=4)

    if len(dgms) == 0:
        ax.text(0.5, 0.5, "No bars", ha="center", va="center",
                transform=ax.transAxes, color="#888", fontsize=8)
        return

    bars = dgms.copy()
    if threshold is not None:
        bars = bars[(bars[:, 1] - bars[:, 0]) > threshold]
    if len(bars) == 0:
        ax.text(0.5, 0.5, "No significant bars", ha="center", va="center",
                transform=ax.transAxes, color="#888", fontsize=8)
        return

    finite = bars[~np.isinf(bars).any(axis=1)]
    if len(finite) == 0:
        return

    xs     = np.linspace(finite.min(), finite.max(), 500)
    deaths = np.where(np.isinf(bars[:, 1]), xs[-1], bars[:, 1])
    curve  = [int(np.sum((bars[:, 0] <= x) & (x <= deaths))) for x in xs]

    ax.fill_between(xs, curve, alpha=0.25, color=color)
    ax.plot(xs, curve, color=color, lw=1.6)
    ax.set_xlabel("Filtration value", fontsize=7)
    ax.set_ylabel("β₁", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.2, lw=0.5)


def plot_block_tda(block_key, ssr_type,
                   dgms1_dict, dgms2_dict,
                   pdb1, pdb2,
                   data_type,
                   methods,
                   out_dir="."):
    """
    For a single structural block plot all TDA methods side by side:
      Row 0: PDB1 persistence diagrams
      Row 1: PDB2 persistence diagrams
      Row 2: PDB1 β₁ curves
      Row 3: PDB2 β₁ curves
    """
    n   = len(methods)
    fig = plt.figure(figsize=(4.5 * n, 14))
    fig.patch.set_facecolor("#1A1F2E")

    fig.suptitle(
        f"{data_type.upper()} TDA  |  Block: {block_key}  |  SSR: {ssr_type}\n"
        f"{'█'*3} {pdb1}   {'█'*3} {pdb2}",
        fontsize=12, fontweight="bold", color="white", y=1.005
    )

    gs = gridspec.GridSpec(4, n, figure=fig, hspace=0.70, wspace=0.42)

    row_labels = [
        f"{pdb1}  —  Persistence Diagram",
        f"{pdb2}  —  Persistence Diagram",
        f"{pdb1}  —  β₁ Curve",
        f"{pdb2}  —  β₁ Curve",
    ]

    for j, method in enumerate(methods):
        dgms1 = dgms1_dict.get(method, np.empty((0, 2)))
        dgms2 = dgms2_dict.get(method, np.empty((0, 2)))
        thr   = THRESHOLDS.get(method)
        lbl   = METHOD_LABELS.get(method, method)

        for row, (dgms, color, rlabel) in enumerate([
            (dgms1, PDB1_COLOR, row_labels[0]),
            (dgms2, PDB2_COLOR, row_labels[1]),
            (dgms1, PDB1_COLOR, row_labels[2]),
            (dgms2, PDB2_COLOR, row_labels[3]),
        ]):
            ax    = fig.add_subplot(gs[row, j])
            title = f"{lbl}\n{rlabel}" if j == 0 else lbl
            if row < 2:
                _plot_pers_diagram(ax, dgms, title, color, thr)
            else:
                _plot_betti_curve(ax, dgms, title, color, thr)
            if j == 0:
                ax.set_ylabel(rlabel + "\nBirth / β₁", fontsize=6.5)

    fname = os.path.join(out_dir, f"tda_{data_type}_block_{block_key}.png")
    plt.savefig(fname, dpi=110, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    plt.close()
    print(f"  [PLOT] {fname}")
    return fname


# ─────────────────────────────────────────────────────────────────────────────
# 8. Final residue vs β₁ plot
# ─────────────────────────────────────────────────────────────────────────────

def plot_residue_vs_betti(all_results, pdb1, pdb2, data_type, methods, out_dir="."):
    """Comparative residue-level β₁ plot for both TDA methods."""
    n    = len(methods)
    fig, axes = plt.subplots(n, 1, figsize=(14, 3.5 * n), sharex=False)
    if n == 1:
        axes = [axes]
    fig.patch.set_facecolor("#12151E")

    fig.suptitle(
        f"{data_type.upper()}  Residue  vs  β₁\n{pdb1}  ●  {pdb2}",
        fontsize=14, fontweight="bold", color="white", y=1.01
    )

    def sorted_keys(pdb_results):
        return sorted(pdb_results.keys(),
                      key=lambda k: int(k.split("_")[2]))

    keys1 = sorted_keys(all_results.get(pdb1, {}))
    keys2 = sorted_keys(all_results.get(pdb2, {}))

    for ax, method in zip(axes, methods):
        ax.set_facecolor("#1E2335")
        ax.tick_params(colors="white", labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor("#444")

        ax.set_title(METHOD_LABELS.get(method, method),
                     fontsize=10, fontweight="bold", color="white", pad=6)
        ax.set_xlabel("Residue", fontsize=8, color="#CCC")
        ax.set_ylabel("β₁",      fontsize=8, color="#CCC")

        def _draw_pdb(pdb, keys, color):
            pdb_res  = all_results.get(pdb, {})
            res_mid  = []
            betti_v  = []
            fills    = []
            for key in keys:
                parts = key.split("_")
                ssr   = parts[1]
                s_res = int(parts[2])
                e_res = int(parts[3])
                b1    = pdb_res.get(key, {}).get(method, 0)
                res_mid.append((s_res + e_res) / 2)
                betti_v.append(b1)
                fills.append((s_res, e_res, ssr))

            for (s, e, ssr) in fills:
                ax.axvspan(s, e, alpha=0.07,
                           color=SSR_COLORS.get(ssr, "#888"), zorder=0)

            if res_mid:
                xs, ys = [], []
                for key, b1 in zip(keys, betti_v):
                    parts = key.split("_")
                    xs += [int(parts[2]), int(parts[3])]
                    ys += [b1, b1]
                ax.step(xs, ys, where="post", color=color, lw=2.0,
                        label=pdb, alpha=0.9, zorder=3)
                ax.fill_between(xs, ys, step="post",
                                color=color, alpha=0.15, zorder=2)
                ax.scatter(res_mid, betti_v, color=color, s=50, zorder=4,
                           edgecolors="white", linewidths=0.5)

        _draw_pdb(pdb1, keys1, PDB1_COLOR)
        _draw_pdb(pdb2, keys2, PDB2_COLOR)

        ssr_patches = [
            Patch(facecolor=SSR_COLORS["AH"], alpha=0.5, label="α-Helix (AH)"),
            Patch(facecolor=SSR_COLORS["BS"], alpha=0.5, label="β-Sheet (BS)"),
            Patch(facecolor=SSR_COLORS["TL"], alpha=0.5, label="Loop / Coil (TL)"),
        ]
        pdb_lines = [
            Line2D([0], [0], color=PDB1_COLOR, lw=2, label=pdb1),
            Line2D([0], [0], color=PDB2_COLOR, lw=2, label=pdb2),
        ]
        ax.legend(handles=pdb_lines + ssr_patches, fontsize=7,
                  loc="upper right", framealpha=0.3,
                  labelcolor="white", facecolor="#2A2F45")

        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.grid(axis="y", alpha=0.15, lw=0.5, color="white")

    plt.tight_layout()
    fname = os.path.join(out_dir, f"residue_vs_betti_{data_type}.png")
    plt.savefig(fname, dpi=120, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    plt.close()
    print(f"\n[FINAL PLOT] {fname}")
    return fname


# ─────────────────────────────────────────────────────────────────────────────
# 9. Summary table
# ─────────────────────────────────────────────────────────────────────────────

def _build_summary_table(betti_dict_pdb1, betti_dict_pdb2, methods,
                          pdb1, pdb2, label="DATA"):
    rows   = []
    keys1  = sorted(betti_dict_pdb1.keys(), key=lambda k: int(k.split("_")[2]))
    keys2  = sorted(betti_dict_pdb2.keys(), key=lambda k: int(k.split("_")[2]))
    n_blk  = min(len(keys1), len(keys2))

    for i in range(n_blk):
        k1, k2 = keys1[i], keys2[i]
        parts  = k1.split("_")
        ssr, s, e = parts[1], parts[2], parts[3]
        for method in methods:
            b1   = betti_dict_pdb1[k1].get(method, 0)
            b2   = betti_dict_pdb2[k2].get(method, 0)
            diff = abs(b1 - b2)
            rows.append({
                "Block"       : i + 1,
                "SSR"         : ssr,
                "Range"       : f"{s}–{e}",
                "Method"      : METHOD_LABELS.get(method, method).replace("\n", " "),
                f"β₁ {pdb1}"  : b1,
                f"β₁ {pdb2}"  : b2,
                "Diff"        : diff,
                "Match"       : "✓ YES" if diff <= 2 else "✗ NO",
            })

    df = pd.DataFrame(rows)
    print(f"\n{'='*90}")
    print(f"  {label} — PERSISTENCE SUMMARY  ({pdb1} vs {pdb2})")
    print(f"{'='*90}")
    print(df.to_string(index=False))
    print(f"{'='*90}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 10. Main analysis
# ─────────────────────────────────────────────────────────────────────────────

def run_full_analysis(pdb1, pdb2, c1=None, c2=None,
                      multiplier=10, out_dir="."):
    """
    Full dual-PDB TDA analysis with Vietoris-Rips (Euclidean) and Alpha Complex.

    Parameters
    ----------
    pdb1, pdb2  : PDB IDs (e.g. "1ABC")
    c1,  c2     : chain IDs (None = first chain)
    multiplier  : GP upsampling factor
    out_dir     : directory for saved plots

    Returns
    -------
    torsion_df  : pd.DataFrame  summary table for torsion methods
    coord_df    : pd.DataFrame  summary table for coord methods
    raw_df1     : pd.DataFrame  raw per-residue table for pdb1
    raw_df2     : pd.DataFrame  raw per-residue table for pdb2
    """
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'#'*80}")
    print(f"  Multi-method TDA  |  {pdb1}  vs  {pdb2}")
    print(f"{'#'*80}")

    # ── Extract, print raw tables, upsample ──────────────────────────────
    t_blocks1, c_blocks1, ranges1, raw_df1 = extract_blocks(pdb1, c1, multiplier)
    t_blocks2, c_blocks2, ranges2, raw_df2 = extract_blocks(pdb2, c2, multiplier)

    # Sort keys by start residue
    t_keys1 = sorted(t_blocks1, key=lambda k: int(k.split("_")[2]))
    t_keys2 = sorted(t_blocks2, key=lambda k: int(k.split("_")[2]))
    c_keys1 = sorted(c_blocks1, key=lambda k: int(k.split("_")[2]))
    c_keys2 = sorted(c_blocks2, key=lambda k: int(k.split("_")[2]))

    torsion_betti = {pdb1: {}, pdb2: {}}
    coord_betti   = {pdb1: {}, pdb2: {}}

    # ── Torsion TDA ───────────────────────────────────────────────────────
    n_t = min(len(t_keys1), len(t_keys2))
    print(f"\n[TORSION TDA] {n_t} blocks")
    for i in range(n_t):
        k1, k2 = t_keys1[i], t_keys2[i]
        ssr    = k1.split("_")[1]
        print(f"\n  Block {i+1}: {k1}  vs  {k2}  (SSR: {ssr})")

        d1 = compute_all_dgms_torsion(t_blocks1[k1])
        d2 = compute_all_dgms_torsion(t_blocks2[k2])

        for method in TORSION_METHODS:
            b1 = betti1_from_dgms(d1[method], THRESHOLDS.get(method))
            b2 = betti1_from_dgms(d2[method], THRESHOLDS.get(method))
            print(f"    {METHOD_LABELS[method].replace(chr(10),' '):42s}  "
                  f"{pdb1}: β₁={b1}   {pdb2}: β₁={b2}   Δ={abs(b1-b2)}")
            torsion_betti[pdb1].setdefault(k1, {})[method] = b1
            torsion_betti[pdb2].setdefault(k2, {})[method] = b2

        plot_block_tda(k1, ssr, d1, d2, pdb1, pdb2,
                       "torsion", TORSION_METHODS, out_dir)

    # ── Coord TDA ─────────────────────────────────────────────────────────
    n_c = min(len(c_keys1), len(c_keys2))
    print(f"\n[COORD TDA] {n_c} blocks")
    for i in range(n_c):
        k1, k2 = c_keys1[i], c_keys2[i]
        ssr    = k1.split("_")[1]
        print(f"\n  Block {i+1}: {k1}  vs  {k2}  (SSR: {ssr})")

        d1 = compute_all_dgms_coords(c_blocks1[k1])
        d2 = compute_all_dgms_coords(c_blocks2[k2])

        for method in COORD_METHODS:
            b1 = betti1_from_dgms(d1[method], THRESHOLDS.get(method))
            b2 = betti1_from_dgms(d2[method], THRESHOLDS.get(method))
            print(f"    {METHOD_LABELS[method].replace(chr(10),' '):42s}  "
                  f"{pdb1}: β₁={b1}   {pdb2}: β₁={b2}   Δ={abs(b1-b2)}")
            coord_betti[pdb1].setdefault(k1, {})[method] = b1
            coord_betti[pdb2].setdefault(k2, {})[method] = b2

        plot_block_tda(k1, ssr, d1, d2, pdb1, pdb2,
                       "coords", COORD_METHODS, out_dir)

    # ── Summary tables ────────────────────────────────────────────────────
    torsion_df = _build_summary_table(
        torsion_betti[pdb1], torsion_betti[pdb2],
        TORSION_METHODS, pdb1, pdb2, "TORSION ANGLES"
    )
    coord_df = _build_summary_table(
        coord_betti[pdb1], coord_betti[pdb2],
        COORD_METHODS, pdb1, pdb2, "Cα COORDINATES"
    )

    # ── Final residue vs β₁ plots ─────────────────────────────────────────
    plot_residue_vs_betti(torsion_betti, pdb1, pdb2,
                          "torsion", TORSION_METHODS, out_dir)
    plot_residue_vs_betti(coord_betti,   pdb1, pdb2,
                          "coords",  COORD_METHODS,   out_dir)

    # ── Print raw DataFrames once more at the end for easy reference ──────
    print(f"\n{'='*90}")
    print(f"  FINAL RAW TABLE  —  {pdb1}")
    print(f"{'='*90}")
    print(raw_df1.to_string(index=False))

    print(f"\n{'='*90}")
    print(f"  FINAL RAW TABLE  —  {pdb2}")
    print(f"{'='*90}")
    print(raw_df2.to_string(index=False))

    return torsion_df, coord_df, raw_df1, raw_df2




In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 1st Example: COVID 19 
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torsion_summary, coord_summary = run_full_analysis(
        pdb1="6VSB",          
        pdb2="7TGW",          
        c1="A",
        c2="A",
        multiplier=10,
        out_dir="./tda_output_covid"
    )


################################################################################
  Multi-method TDA  |  6VSB  vs  7TGW
################################################################################
  Note: Accessing on-line PDB file

  RAW EXTRACTION TABLE  —  6VSB
  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR
   residue  resno_int      phi      psi     chi1     chi2     chi3     chi4  chi5    omega    alpha    ca_x    ca_y    ca_z SSR
  A/27/ALA         27 0.500000 0.832233 0.500000 0.500000 0.500000 0.500000   0.5 0.048140 0.500000 170.347 220.661 242.750  TL
  A/28/TYR         28 0.164198 0.943416 0.262272 0.467909 0.500000 0.500000   0.5 0.049914 0.055189 173.409 221.392 240.626  BS
  A/29/THR         29 0.071656 0.940144 0.295775 0.500000 0.500000 0.500000   0.5 0.049455 0.167006 174.069 224.295 238.262  BS
  A/30/ASN         30 0.168036 0.858670 0.164431 0.307234 0.500000 0.500000   0.5 0.945792 0.066890 176.576 225.424 235.618  BS
  

ValueError: too many values to unpack (expected 2)

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 2nd Example : HCCAA (Stabilization)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torsion_summary, coord_summary, raw1, raw2 = run_full_analysis(
        pdb1="1TIJ",           
        pdb2="3GAX",           
        c1="A",
        c2="A",
        multiplier=10,
        out_dir="./tda_output_1tij"
    )



################################################################################
  Multi-method TDA  |  1TIJ  vs  3GAX
################################################################################
  Note: Accessing on-line PDB file

  RAW EXTRACTION TABLE  —  1TIJ
  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR
  residue  resno_int      phi      psi     chi1     chi2     chi3     chi4  chi5    omega    alpha    ca_x   ca_y    ca_z SSR
  A/7/PRO          7 0.500000 0.944215 0.357304 0.664425 0.500000 0.500000   0.5 0.048308 0.500000 110.736 67.299 128.149  TL
  A/8/ARG          8 0.086328 0.839122 0.951495 0.061075 0.705029 0.060924   0.5 0.048055 0.071006 109.047 70.721 128.025  TL
  A/9/LEU          9 0.355120 0.248239 0.345520 0.938476 0.500000 0.500000   0.5 0.952252 0.488898 106.656 71.258 125.087  TL
 A/10/VAL         10 0.086916 0.874087 0.736298 0.500000 0.500000 0.500000   0.5 0.951435 0.912941 107.037 75.035 125.052  BS
 A/11/GLY   

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 3rd Example : Metabolic Stability (L99A Substitution)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torsion_summary, coord_summary = run_full_analysis(
        pdb1="2LZM",           # e.g. Ubiquitin
        pdb2="1L63",           # replace with second PDB
        c1="A",
        c2="A",
        multiplier=10,
        out_dir="./tda_output_1L63"
    )



################################################################################
  Multi-method TDA  |  2LZM  vs  1L63
################################################################################
  Note: Accessing on-line PDB file

  RAW EXTRACTION TABLE  —  2LZM
  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR
  residue  resno_int      phi      psi     chi1     chi2     chi3     chi4  chi5    omega    alpha   ca_x   ca_y    ca_z SSR
  A/1/MET          1 0.500000 0.923031 0.950489 0.879065 0.290670 0.500000   0.5 0.047645 0.500000 43.783 -1.951   8.984  TL
  A/2/ASN          2 0.082049 0.945848 0.731893 0.716856 0.500000 0.500000   0.5 0.047529 0.193357 40.653 -0.795  10.918  TL
  A/3/ILE          3 0.295736 0.338100 0.057434 0.719426 0.500000 0.500000   0.5 0.047842 0.700569 38.546  2.322  11.353  AH
  A/4/PHE          4 0.240221 0.329520 0.250786 0.821870 0.500000 0.500000   0.5 0.951515 0.679285 40.752  3.772  14.110  AH
  A/5/GLU        

ValueError: too many values to unpack (expected 2)

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 4th Example : Sickle Cell Anemia
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torsion_summary, coord_summary = run_full_analysis(
        pdb1="9O0N",          
        pdb2="4LDJ",          
        c1="A",
        c2="A",
        multiplier=10,
        out_dir="./tda_output_9O0N"
    )



################################################################################
  Multi-method TDA  |  9O0N  vs  4LDJ
################################################################################
  Note: Accessing on-line PDB file
   PDB has ALT records, taking A only, rm.alt=TRUE

  RAW EXTRACTION TABLE  —  9O0N
  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR
  residue  resno_int      phi      psi     chi1     chi2     chi3     chi4  chi5    omega    alpha   ca_x    ca_y   ca_z SSR
  A/1/MET          1 0.500000 0.910873 0.062295 0.060361 0.197136 0.500000   0.5 0.050449 0.500000  1.559  17.138 14.312  TL
  A/2/THR          2 0.247049 0.892890 0.290327 0.500000 0.500000 0.500000   0.5 0.952378 0.056987  2.387  13.496 13.588  BS
  A/3/GLU          3 0.121067 0.912690 0.053229 0.164808 0.507102 0.500000   0.5 0.952154 0.064824  5.394  12.265 15.527  BS
  A/4/TYR          4 0.113614 0.897569 0.254529 0.200581 0.500000 0.500000   0.5 0.952459 0

ValueError: too many values to unpack (expected 2)

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 5th Example : Human Cancer (Core Mutation)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torsion_summary, coord_summary = run_full_analysis(
        pdb1="1TUP",          
        pdb2="4MZI",          
        c1="A",
        c2="A",
        multiplier=10,
        out_dir="./tda_output_1TUP"
    )


################################################################################
  Multi-method TDA  |  1TUP  vs  4MZI
################################################################################
  Note: Accessing on-line PDB file

  RAW EXTRACTION TABLE  —  1TUP
  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR
  residue  resno_int      phi      psi     chi1     chi2     chi3     chi4  chi5    omega    alpha   ca_x   ca_y   ca_z SSR
 A/94/SER         94 0.500000 0.945544 0.235649 0.500000 0.500000 0.500000   0.5 0.951302 0.500000 75.562 21.797 80.653  TL
 A/95/SER         95 0.283397 0.895426 0.119855 0.500000 0.500000 0.500000   0.5 0.952214 0.131000 74.668 19.703 77.670  TL
 A/96/SER         96 0.248483 0.924731 0.599131 0.500000 0.500000 0.500000   0.5 0.048348 0.050424 77.257 20.526 75.012  TL
 A/97/VAL         97 0.087749 0.938778 0.194082 0.500000 0.500000 0.500000   0.5 0.952135 0.234136 76.057 22.298 71.922  TL
 A/98/PRO         98 0

ValueError: too many values to unpack (expected 2)

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 6th Example : Human Cancer (Scaffold Mutation)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torsion_summary, coord_summary = run_full_analysis(
        pdb1="1AUP",          
        pdb2="1K89",          
        c1="A",
        c2="A",
        multiplier=10,
        out_dir="./tda_output_1AUP"
    )


################################################################################
  Multi-method TDA  |  1AUP  vs  1K89
################################################################################
  Note: Accessing on-line PDB file

  RAW EXTRACTION TABLE  —  1AUP
  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR
  residue  resno_int      phi      psi     chi1     chi2     chi3     chi4  chi5    omega    alpha    ca_x   ca_y   ca_z SSR
  A/1/SER          1 0.500000 0.908307 0.047645 0.500000 0.500000 0.500000   0.5 0.950906 0.500000   0.399 43.017 18.010  TL
  A/2/LYS          2 0.255672 0.328888 0.251293 0.924924 0.055050 0.188036   0.5 0.949472 0.694017   0.764 44.436 14.560  AH
  A/3/TYR          3 0.280111 0.285110 0.948055 0.609014 0.500000 0.500000   0.5 0.047497 0.681779   1.555 40.901 13.104  AH
  A/4/VAL          4 0.287955 0.302626 0.943314 0.500000 0.500000 0.500000   0.5 0.049090 0.691856  -1.483 39.580 14.928  AH
  A/5/ASP        

ValueError: too many values to unpack (expected 2)

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 7th Example : Human Cancer(Scaffold Mutation)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torsion_summary, coord_summary = run_full_analysis(
        pdb1="1AUP",           
        pdb2="1k89",           
        c1="A",
        c2="A",
        multiplier=10,
        out_dir="./tda_output_GDH"
    )


################################################################################
  Multi-method TDA  |  1AUP  vs  1k89
################################################################################
  Note: Accessing on-line PDB file

  RAW EXTRACTION TABLE  —  1AUP
  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR
  residue  resno_int      phi      psi     chi1     chi2     chi3     chi4  chi5    omega    alpha    ca_x   ca_y   ca_z SSR
  A/1/SER          1 0.500000 0.908307 0.047645 0.500000 0.500000 0.500000   0.5 0.950906 0.500000   0.399 43.017 18.010  TL
  A/2/LYS          2 0.255672 0.328888 0.251293 0.924924 0.055050 0.188036   0.5 0.949472 0.694017   0.764 44.436 14.560  AH
  A/3/TYR          3 0.280111 0.285110 0.948055 0.609014 0.500000 0.500000   0.5 0.047497 0.681779   1.555 40.901 13.104  AH
  A/4/VAL          4 0.287955 0.302626 0.943314 0.500000 0.500000 0.500000   0.5 0.049090 0.691856  -1.483 39.580 14.928  AH
  A/5/ASP        

ValueError: too many values to unpack (expected 2)

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 8th Example: Rhabdomyolysis(Denaturation)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torsion_summary, coord_summary = run_full_analysis(
        pdb1="1WLA",           
        pdb2="1ABS",           
        c1="A",
        c2="A",
        multiplier=10,
        out_dir="./tda_output_1WLA"
    )



################################################################################
  Multi-method TDA  |  1WLA  vs  1ABS
################################################################################
  Note: Accessing on-line PDB file

  RAW EXTRACTION TABLE  —  1WLA
  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR
  residue  resno_int      phi      psi     chi1     chi2     chi3     chi4  chi5    omega    alpha   ca_x   ca_y   ca_z SSR
  A/1/GLY          1 0.500000 0.349406 0.500000 0.500000 0.500000 0.500000   0.5 0.952393 0.500000 -2.860 16.837 15.206  TL
  A/2/LEU          2 0.127987 0.930524 0.181070 0.777239 0.500000 0.500000   0.5 0.948818 0.110183 -0.557 15.141 17.748  TL
  A/3/SER          3 0.192063 0.943459 0.760928 0.500000 0.500000 0.500000   0.5 0.952465 0.141212 -1.311 13.550 21.145  AH
  A/4/ASP          4 0.240398 0.364633 0.075988 0.195565 0.500000 0.500000   0.5 0.952124 0.724851  0.654 14.785 24.202  AH
  A/5/GLY          5 0

ValueError: too many values to unpack (expected 2)

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 9th Example : Skin Cancer(Loop Alteration)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torsion_summary, coord_summary = run_full_analysis(
        pdb1="1CBI",           
        pdb2="1CBQ",          
        c1="A",
        c2="A",
        multiplier=10,
        out_dir="./tda_output_1CBI"
    )



################################################################################
  Multi-method TDA  |  1CBI  vs  1CBQ
################################################################################
  Note: Accessing on-line PDB file

  RAW EXTRACTION TABLE  —  1CBI
  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR
  residue  resno_int      phi      psi     chi1     chi2     chi3     chi4  chi5    omega    alpha    ca_x    ca_y   ca_z SSR
  A/1/PRO          1 0.500000 0.952507 0.566536 0.365057 0.500000 0.500000   0.5 0.952538 0.500000   7.699 104.034 31.626  TL
  A/2/ASN          2 0.128765 0.061340 0.728140 0.528850 0.500000 0.500000   0.5 0.047896 0.207456   4.573 103.458 29.522  TL
  A/3/PHE          3 0.208717 0.449455 0.241929 0.767878 0.500000 0.500000   0.5 0.048898 0.875456   2.713 100.270 28.510  TL
  A/4/ALA          4 0.262360 0.912360 0.500000 0.500000 0.500000 0.500000   0.5 0.951427 0.719733   5.123  99.438 25.679  TL
  A/5/GLY   

ValueError: too many values to unpack (expected 2)

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# 10th Example : Carney Complex(Loop Mutation)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    torsion_summary, coord_summary = run_full_analysis(
        pdb1="1ATP",          
        pdb2="1U7E",          
        c1="E",
        c2="A",
        multiplier=10,
        out_dir="./tda_output_1ATP"
    )


################################################################################
  Multi-method TDA  |  1ATP  vs  1U7E
################################################################################
  Note: Accessing on-line PDB file

  RAW EXTRACTION TABLE  —  1ATP
  Columns: chain/resno/resname (e.g. A/42/SER) | torsion angles (sigmoid) | Cα XYZ | SSR
  residue  resno_int      phi      psi     chi1     chi2     chi3     chi4  chi5    omega    alpha   ca_x    ca_y    ca_z SSR
 E/15/VAL         15 0.500000 0.254124 0.933660 0.500000 0.500000 0.500000   0.5 0.951236 0.500000 -5.276 -11.431 -13.476  TL
 E/16/LYS         16 0.300479 0.346039 0.076179 0.835628 0.211546 0.864523   0.5 0.951802 0.720668 -4.060 -11.108 -17.074  AH
 E/17/GLU         17 0.290735 0.262426 0.331314 0.847265 0.691547 0.500000   0.5 0.047693 0.673908 -7.452  -9.768 -17.502  AH
 E/18/PHE         18 0.301396 0.266677 0.949157 0.735225 0.500000 0.500000   0.5 0.952539 0.703503 -7.050  -7.008 -14.884  AH
 E/19/LEU   

ValueError: too many values to unpack (expected 2)